# Week 1: Diffusion Policy on PushT — From Scratch to Mastery

**Vizuara Robot Learning Bootcamp**

In this notebook, you will:
1. Understand the PushT environment and what we're trying to solve
2. Explore the demonstration dataset — what does expert data look like?
3. Build deep intuition for every component of Diffusion Policy:
   - ResNet18 vision encoder
   - SpatialSoftmax keypoint extraction
   - Sinusoidal timestep embeddings
   - Global conditioning vector construction
   - 1D U-Net with FiLM conditioning
   - DDPM forward & reverse diffusion
4. Train a Diffusion Policy from scratch
5. Evaluate it in simulation and watch it solve PushT
6. Visualize what the model has learned
7. Run ablation experiments to understand what matters

> **Hardware**: This notebook is designed for Google Colab with a T4 GPU (free tier works).

---

## Part 0: Setup & Installation

We install LeRobot (HuggingFace's robot learning library) and gym-pusht (the PushT simulation environment).

In [ ]:
# Install LeRobot with PushT environment support
!pip install -q "lerobot[pusht]" mediapy 2>&1 | tail -5

# Verify installations
import lerobot
import gym_pusht
print(f"LeRobot version: {lerobot.__version__}")
print("gym_pusht: OK")

In [ ]:
# Core imports we'll use throughout
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import HTML
import mediapy as media
import gymnasium as gym

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Part 1: Meet the PushT Environment

**The Task**: A blue circle (the agent) must push a light-gray T-shaped block onto a green target zone. The episode is "solved" when the T-block overlaps the target by ≥ 95%.

**Why PushT?** It looks simple, but it's deceptively rich:
- There are **multiple valid strategies** to push the T onto the target (approach from left, right, top...)
- This **multimodality** is exactly what makes naive regression (MSE) policies fail — they average the modes and produce nonsense
- Diffusion Policy handles this naturally by sampling from the full distribution of valid trajectories

**Observation space:**
- `pixels`: 96×96 RGB image of the scene
- `agent_pos`: (x, y) position of the blue agent circle

**Action space:**
- 2D: (x, y) target position for the agent to move toward

In [ ]:
# Create the PushT environment and see what it looks like
env = gym.make(
    "gym_pusht/PushT-v0",
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
)

obs, info = env.reset(seed=42)

print("=== Observation Space ===")
print(f"  pixels shape : {obs['pixels'].shape}")     # (384, 384, 3)
print(f"  pixels dtype : {obs['pixels'].dtype}")      # uint8
print(f"  pixels range : [{obs['pixels'].min()}, {obs['pixels'].max()}]")
print(f"  agent_pos    : {obs['agent_pos']}")          # [x, y]
print(f"\n=== Action Space ===")
print(f"  shape: {env.action_space.shape}")           # (2,)
print(f"  low  : {env.action_space.low}")
print(f"  high : {env.action_space.high}")

# Visualize the initial state
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(obs['pixels'])
axes[0].set_title("What the camera sees (384×384)")
axes[0].axis('off')

# Show the agent position overlaid
axes[1].imshow(obs['pixels'])
ax, ay = obs['agent_pos']
axes[1].plot(ax, ay, 'ro', markersize=15, markeredgecolor='white', markeredgewidth=2)
axes[1].set_title(f"Agent position: ({ax:.0f}, {ay:.0f})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Let's watch a RANDOM policy — it will fail miserably!
# This shows why we need learned policies.

obs, info = env.reset(seed=42)
frames = [env.render()]

total_reward = 0
for step in range(100):  # Short rollout
    action = env.action_space.sample()  # Random action!
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if step % 2 == 0:  # Save every other frame to keep video small
        frames.append(env.render())

env.close()

print(f"Random policy total reward: {total_reward:.2f}")
print(f"Saved {len(frames)} frames")

# Display as video
media.show_video(frames, fps=10, title="Random Policy (it's bad!)")

**Observation**: The random policy just jitters around aimlessly. It never coordinates to push the T-block toward the target. We need to learn from demonstrations!

---
## Part 2: Exploring the Demonstration Dataset

LeRobot provides 205 expert demonstration trajectories for PushT, collected via human teleoperation.

Each trajectory is a sequence of (observation, action) pairs at 10 FPS.

### Key Concept: `delta_timestamps`

Diffusion Policy doesn't just look at the current frame. It uses **temporal context**:
- **Observations**: frames at `t-1` and `t` (2 frames of history)
- **Actions**: a horizon of 16 future actions from `t-1` to `t+14`

The `delta_timestamps` tell LeRobot exactly which timesteps to load relative to the current one.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

# First, load just the metadata (fast — no data download yet)
metadata = LeRobotDatasetMetadata("lerobot/pusht")

print("=== Dataset Metadata ===")
print(f"  FPS: {metadata.fps}")
print(f"  Total episodes: {metadata.total_episodes}")
print(f"  Total frames: {metadata.total_frames}")
print(f"\n=== Features ===")
for name, feat in metadata.features.items():
    print(f"  {name}: {feat}")

In [ ]:
# Now load the full dataset with temporal windowing
# This is the EXACT windowing Diffusion Policy uses

delta_timestamps = {
    # Load 2 consecutive images: t-1 and t
    # At 10 FPS, one frame = 0.1 seconds
    "observation.image": [-0.1, 0.0],

    # Load agent state (x,y position) at same 2 timesteps
    "observation.state": [-0.1, 0.0],

    # Load 16 actions: from t-1 to t+14
    # This is the "action horizon" — we predict all 16 but only execute 8
    "action": [-0.1, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4],
}

dataset = LeRobotDataset("lerobot/pusht", delta_timestamps=delta_timestamps)
print(f"\nDataset size: {len(dataset)} samples")

In [ ]:
# Let's inspect a single training sample in detail
sample = dataset[1000]  # Pick a sample from the middle

print("=== Single Training Sample ===")
for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key:30s} shape={str(val.shape):15s} dtype={val.dtype}")
    else:
        print(f"  {key:30s} = {val}")

In [ ]:
# Visualize: what does the model see vs what it must predict?

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# The two observation images (t-1 and t)
img_t_minus_1 = sample['observation.image'][0].permute(1, 2, 0).numpy()  # [C,H,W] -> [H,W,C]
img_t = sample['observation.image'][1].permute(1, 2, 0).numpy()

axes[0].imshow(img_t_minus_1)
axes[0].set_title("Observation at t-1")
axes[0].axis('off')

axes[1].imshow(img_t)
axes[1].set_title("Observation at t (current)")
axes[1].axis('off')

# The 16 target actions overlaid on the current image
actions = sample['action'].numpy()  # [16, 2]
axes[2].imshow(img_t)
# Scale actions to image coordinates (actions are in environment coords)
# Plot the trajectory of actions
axes[2].plot(actions[:, 0], actions[:, 1], 'r.-', linewidth=2, markersize=8, alpha=0.8)
axes[2].plot(actions[0, 0], actions[0, 1], 'go', markersize=12, label='Start')
axes[2].plot(actions[-1, 0], actions[-1, 1], 'rs', markersize=12, label='End')
axes[2].set_title(f"Target: 16 future actions\n(predict all, execute first 8)")
axes[2].legend()
axes[2].axis('off')

plt.suptitle("What Diffusion Policy Sees (left, center) vs Must Predict (right)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nAgent state at t-1: {sample['observation.state'][0].numpy()}")
print(f"Agent state at t  : {sample['observation.state'][1].numpy()}")
print(f"Action horizon shape: {actions.shape} (16 actions × 2 coords)")

In [ ]:
# Visualize multiple expert trajectories to see the MULTIMODALITY
# This is the key insight — same start state, multiple valid solutions!

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Sample 8 different trajectories
indices = np.linspace(0, len(dataset)-1, 8, dtype=int)

for i, idx in enumerate(indices):
    ax = axes[i // 4, i % 4]
    s = dataset[idx]
    img = s['observation.image'][1].permute(1, 2, 0).numpy()
    acts = s['action'].numpy()

    ax.imshow(img)
    ax.plot(acts[:, 0], acts[:, 1], 'r.-', linewidth=2, markersize=4)
    ax.set_title(f"Episode sample {idx}", fontsize=10)
    ax.axis('off')

plt.suptitle("Expert Demonstrations: Different strategies for different states", fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 3: Architecture Deep Dive — How Diffusion Policy Works

Now let's build the Diffusion Policy and inspect every single component.

### The Big Picture

```
INPUTS                              PROCESSING                        OUTPUT
──────                              ──────────                        ──────
2 RGB images (96×96)  ──► ResNet18 + SpatialSoftmax ──► 128d ─┐
                                                               ├─► concat ──► 260d global_cond
Agent state (x,y × 2) ──────────────────────────────► 4d   ─┤                    │
                                                               │                    │
Diffusion timestep k  ──► Sinusoidal + MLP ──────────► 128d ─┘                    │
                                                                                    ▼
                                                                            ┌─── 1D U-Net ───┐
Noisy action trajectory ──────────────────────────────────────────────────►│  FiLM at every  │
(16 × 2 = 32d)                                                             │  layer          │
                                                                            └────────┬────────┘
                                                                                     │
                                                                              Predicted noise
                                                                              (16 × 2 = 32d)
```

Let's instantiate the policy and inspect each component.

In [ ]:
from lerobot.configs.types import FeatureType
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

# Create the policy configuration
features = dataset_to_policy_features(metadata.features)
output_features = {k: ft for k, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {k: ft for k, ft in features.items() if k not in output_features}

cfg = DiffusionConfig(input_features=input_features, output_features=output_features)

print("=== Key Configuration Values ===")
print(f"  n_obs_steps      = {cfg.n_obs_steps}         # Number of observation frames")
print(f"  horizon          = {cfg.horizon}            # Total action prediction length")
print(f"  n_action_steps   = {cfg.n_action_steps}          # Actions actually executed")
print(f"  vision_backbone  = {cfg.vision_backbone}")
print(f"  num_keypoints    = {cfg.spatial_softmax_num_keypoints}")
print(f"  timestep_emb_dim = {cfg.diffusion_step_embed_dim}")
print(f"  down_dims        = {cfg.down_dims}")
print(f"  noise_scheduler  = {cfg.noise_scheduler_type}")
print(f"  train_timesteps  = {cfg.num_train_timesteps}")
print(f"  prediction_type  = {cfg.prediction_type}")
print(f"  use_film_scale   = {cfg.use_film_scale_modulation}")
print(f"  beta_schedule    = {cfg.beta_schedule}")

In [ ]:
# Instantiate the full policy
policy = DiffusionPolicy(cfg)
policy.to(device)

# Count parameters
total_params = sum(p.numel() for p in policy.parameters())
trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable parameters: {trainable_params:,}")

### 3A: The Vision Encoder — ResNet18 + SpatialSoftmax

**Goal**: Turn a 96×96 RGB image into a compact vector of spatial keypoints.

**Pipeline**:
1. **ResNet18** (minus classification head) → produces a feature map of shape `(512, H', W')`
2. **Conv2d(512 → 32)** → reduces to 32 channels: `(32, H', W')`
3. **SpatialSoftmax** → for each channel, compute the (x, y) center of mass → `(32, 2)` = 64 floats
4. **Linear(64 → 64) + ReLU** → final feature vector

The SpatialSoftmax is the key innovation here: instead of global average pooling (which loses spatial info), it tells us **WHERE** each feature is active.

In [ ]:
# Let's trace through the vision encoder step by step
rgb_encoder = policy.diffusion.rgb_encoder
print("=== RGB Encoder Architecture ===")
print(rgb_encoder)
print(f"\nOutput feature dim: {rgb_encoder.feature_dim}")  # 64

In [ ]:
# Step-by-step forward pass through the encoder
# Using a real image from the dataset

sample = dataset[1000]
test_image = sample['observation.image'][1:2].to(device)  # [1, 3, 96, 96] - single image
print(f"Input image shape: {test_image.shape}")
print(f"Input pixel range: [{test_image.min():.3f}, {test_image.max():.3f}]")

# Step 1: ResNet18 backbone (minus final layers)
with torch.no_grad():
    backbone_features = rgb_encoder.backbone(test_image)
print(f"\nStep 1 - ResNet18 backbone output: {backbone_features.shape}")
# Expected: [1, 512, H', W'] where H',W' depends on input size

# Step 2: SpatialSoftmax
with torch.no_grad():
    keypoints = rgb_encoder.pool(backbone_features)
print(f"Step 2 - SpatialSoftmax output: {keypoints.shape}")
# Expected: [1, 32, 2] — 32 keypoints, each with (x, y)

# Step 3: Flatten + Linear + ReLU
with torch.no_grad():
    flat_keypoints = torch.flatten(keypoints, start_dim=1)  # [1, 64]
    final_features = rgb_encoder.relu(rgb_encoder.out(flat_keypoints))
print(f"Step 3 - Final feature vector: {final_features.shape}")
# Expected: [1, 64]

# Verify end-to-end
with torch.no_grad():
    e2e_features = rgb_encoder(test_image)
print(f"\nEnd-to-end output: {e2e_features.shape}")
print(f"Match: {torch.allclose(final_features, e2e_features)}")

In [ ]:
# Deep dive: How SpatialSoftmax actually works
# This is the most important thing to understand!

pool = rgb_encoder.pool

print("=== SpatialSoftmax Internals ===")
print(f"Input channels (after conv projection): {pool._out_c}")
print(f"Spatial grid size: {pool._in_h} × {pool._in_w}")
print(f"Coordinate grid shape: {pool.pos_grid.shape}")  # [H*W, 2]
print(f"\nCoordinate grid (first 5 positions):")
print(pool.pos_grid[:5].cpu().numpy())
print(f"\nCoordinate grid (x range): [{pool.pos_grid[:, 0].min():.2f}, {pool.pos_grid[:, 0].max():.2f}]")
print(f"Coordinate grid (y range): [{pool.pos_grid[:, 1].min():.2f}, {pool.pos_grid[:, 1].max():.2f}]")

# Manual SpatialSoftmax computation for ONE channel
print("\n=== Manual Computation for Channel 0 ===")
with torch.no_grad():
    # Get feature maps after the learnable projection
    projected = pool.nets(backbone_features)  # [1, 32, H, W]
    channel_0 = projected[0, 0]  # [H, W] — activations of channel 0
    print(f"Channel 0 activations shape: {channel_0.shape}")
    print(f"Channel 0 raw values: min={channel_0.min():.3f}, max={channel_0.max():.3f}")

    # Flatten and apply softmax
    flat_activations = channel_0.reshape(-1)  # [H*W]
    attention_weights = torch.softmax(flat_activations, dim=0)  # [H*W]
    print(f"\nAfter softmax (attention weights):")
    print(f"  Sum = {attention_weights.sum():.4f} (should be 1.0)")
    print(f"  Max weight = {attention_weights.max():.4f}")
    print(f"  Where is max? Position {attention_weights.argmax().item()}")

    # Weighted average of coordinates = expected keypoint position
    expected_x = (attention_weights @ pool.pos_grid[:, 0]).item()
    expected_y = (attention_weights @ pool.pos_grid[:, 1]).item()
    print(f"\n  Keypoint 0 position: ({expected_x:.4f}, {expected_y:.4f})")
    print(f"  (range is [-1, 1] — normalized image coordinates)")

In [ ]:
# Visualize: where are the 32 keypoints on the image?

with torch.no_grad():
    kps = rgb_encoder.pool(backbone_features)  # [1, 32, 2]
    kps = kps[0].cpu().numpy()  # [32, 2]

img = sample['observation.image'][1].permute(1, 2, 0).numpy()
h, w = img.shape[:2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original image
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[0].axis('off')

# Image with keypoints overlaid
axes[1].imshow(img)
# Convert keypoints from [-1, 1] to pixel coordinates
pixel_x = (kps[:, 0] + 1) / 2 * w
pixel_y = (kps[:, 1] + 1) / 2 * h
axes[1].scatter(pixel_x, pixel_y, c='red', s=80, marker='x', linewidths=2, zorder=5)
for i in range(len(kps)):
    axes[1].annotate(str(i), (pixel_x[i]+2, pixel_y[i]+2), fontsize=6, color='yellow')
axes[1].set_title(f"32 SpatialSoftmax Keypoints\n(where the model 'looks')")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("Each keypoint is a (x,y) coordinate telling the policy")
print("WHERE a particular visual feature is most active.")
print("Notice: keypoints cluster on the T-block and agent — the task-relevant objects!")

### 3B: The Diffusion Timestep Embedding

During denoising, the U-Net needs to know **which noise level** it's currently working at. Step 99 (very noisy) requires different processing than step 1 (almost clean).

The timestep `k` is encoded as:
1. **Sinusoidal positional encoding** (same idea as Transformers) → 128 dims
2. **Linear(128 → 512) + Mish + Linear(512 → 128)** → learned transformation

This gives each noise level a unique, smooth fingerprint.

In [ ]:
# Inspect the timestep encoder
timestep_encoder = policy.diffusion.unet.diffusion_step_encoder
print("=== Timestep Encoder ===")
print(timestep_encoder)

# Compute embeddings for all 100 timesteps
with torch.no_grad():
    all_timesteps = torch.arange(100, device=device)
    embeddings = timestep_encoder(all_timesteps)  # [100, 128]
    print(f"\nEmbedding shape for all timesteps: {embeddings.shape}")

# Visualize: how different are nearby vs distant timesteps?
embeddings_cpu = embeddings.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Heatmap of all embeddings
axes[0].imshow(embeddings_cpu, aspect='auto', cmap='viridis')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Diffusion timestep')
axes[0].set_title('All 100 timestep embeddings (128-dim each)')

# Compare specific timesteps
for t in [0, 25, 50, 75, 99]:
    axes[1].plot(embeddings_cpu[t, :20], label=f't={t}')  # First 20 dims
axes[1].set_xlabel('Embedding dimension (first 20)')
axes[1].set_ylabel('Value')
axes[1].set_title('Embedding values at different timesteps')
axes[1].legend()

# Cosine similarity between timesteps
from torch.nn.functional import cosine_similarity
sim_matrix = np.zeros((100, 100))
for i in range(100):
    for j in range(100):
        sim_matrix[i, j] = cosine_similarity(
            embeddings[i:i+1], embeddings[j:j+1]
        ).item()
axes[2].imshow(sim_matrix, cmap='RdYlBu_r')
axes[2].set_xlabel('Timestep')
axes[2].set_ylabel('Timestep')
axes[2].set_title('Cosine similarity between timestep embeddings')
plt.colorbar(axes[2].images[0], ax=axes[2])

plt.tight_layout()
plt.show()

print("\nNotice: nearby timesteps have similar embeddings (smooth gradient on diagonal).")
print("This smoothness helps the U-Net generalize across noise levels.")

### 3C: The Global Conditioning Vector

Everything the U-Net needs to know about the current situation is packed into one vector:

| Component | Dims | Source |
|-----------|------|--------|
| Image features (camera 1, t-1) | 64 | ResNet18 → SpatialSoftmax → Linear |
| Image features (camera 1, t) | 64 | Same encoder, different frame |
| Robot state (t-1) | 2 | Agent (x,y) position |
| Robot state (t) | 2 | Agent (x,y) position |
| **Total observation cond** | **132** | Flattened across time |
| Diffusion timestep | 128 | Sinusoidal → MLP |
| **Grand total (FiLM cond)** | **260** | Concatenated in U-Net forward |

This 260-dim vector is injected into **every layer** of the U-Net via FiLM conditioning.

In [ ]:
# Let's manually build the global conditioning vector
import einops

# Prepare a batch (mimicking what happens during inference)
sample = dataset[1000]
batch = {
    'observation.state': sample['observation.state'].unsqueeze(0).to(device),        # [1, 2, 2]
    'observation.images': sample['observation.image'].unsqueeze(0).unsqueeze(2).to(device),  # [1, 2, 1, 3, H, W]
}

print("=== Building Global Conditioning ===")
print(f"State shape: {batch['observation.state'].shape}")
print(f"Images shape: {batch['observation.images'].shape}")

with torch.no_grad():
    # Step 1: State features
    state = batch['observation.state']  # [1, 2, 2]
    print(f"\n1. State features: {state.shape} → will be flattened to {state.shape[0]}×{state[0].numel()} = {state[0].numel()}d")

    # Step 2: Image features
    images = batch['observation.images']  # [1, 2, 1, 3, H, W]
    B, S, N = images.shape[:3]
    images_flat = einops.rearrange(images, "b s n ... -> (b s n) ...")  # [2, 3, H, W]
    img_feats = rgb_encoder(images_flat)  # [2, 64]
    img_feats = einops.rearrange(img_feats, "(b s n) ... -> b s (n ...)", b=B, s=S)  # [1, 2, 64]
    print(f"2. Image features: {img_feats.shape} → per-frame={img_feats.shape[-1]}d, total={img_feats[0].numel()}d")

    # Step 3: Concatenate and flatten
    global_cond_parts = [state, img_feats]
    global_cond = torch.cat(global_cond_parts, dim=-1).flatten(start_dim=1)
    print(f"3. Concatenated & flattened: {global_cond.shape}")
    print(f"   = state({state[0].numel()}) + images({img_feats[0].numel()}) = {global_cond.shape[-1]}d")

    # Step 4: In the U-Net, this gets concatenated with timestep embedding
    t_emb = timestep_encoder(torch.tensor([50], device=device))  # [1, 128]
    film_cond = torch.cat([t_emb, global_cond], dim=-1)
    print(f"\n4. Final FiLM conditioning = timestep({t_emb.shape[-1]}) + obs({global_cond.shape[-1]}) = {film_cond.shape[-1]}d")
    print(f"   This {film_cond.shape[-1]}d vector is injected into EVERY U-Net layer!")

### 3D: The 1D U-Net with FiLM Conditioning

The U-Net is the workhorse. It takes a **noisy action trajectory** and predicts the noise to remove.

**Key insight**: This is a **1D** U-Net (not 2D like in image diffusion). The input is a 1D sequence of 16 actions, each with 2 coordinates.

**Architecture**:
```
Input: [B, 16, 2] noisy actions → rearrange to [B, 2, 16] (channels first)

Encoder:                              Decoder:
  [B, 2, 16]                            [B, 2048, 4] ──────── skip ── [B, 2048, 4]
     │ ResBlock + FiLM                     │ ResBlock + FiLM
  [B, 512, 16]  ──── skip ───────────►  [B, 1024, 8]
     │ Downsample                          │ Upsample
  [B, 512, 8]                           [B, 512, 16]
     │ ResBlock + FiLM                     │ ResBlock + FiLM
  [B, 1024, 8]  ──── skip ───────────►  [B, 512, 16]
     │ Downsample                          │
  [B, 1024, 4]                          Final Conv → [B, 2, 16]
     │ ResBlock + FiLM                  Rearrange → [B, 16, 2]
  [B, 2048, 4]
     │
   Middle blocks
```

**FiLM conditioning** (Feature-wise Linear Modulation):
At each residual block, the 260-dim conditioning vector produces per-channel scale and bias:
```python
scale, bias = Linear(conditioning_vector)  # → [B, channels, 1]
hidden = scale * hidden + bias             # Affine transform per channel
```

In [ ]:
# Inspect the U-Net architecture
unet = policy.diffusion.unet

print("=== U-Net Architecture ===")
print(f"\n--- Encoder (Down) Modules ---")
for i, module in enumerate(unet.down_modules):
    resnet1, resnet2, downsample = module
    print(f"  Block {i}: {resnet1.conv1.block[0].in_channels} → {resnet1.conv1.block[0].out_channels}")
    print(f"           Downsample: {type(downsample).__name__}")

print(f"\n--- Middle Modules ---")
for i, module in enumerate(unet.mid_modules):
    print(f"  Block {i}: {module.conv1.block[0].in_channels} → {module.conv1.block[0].out_channels}")

print(f"\n--- Decoder (Up) Modules ---")
for i, module in enumerate(unet.up_modules):
    resnet1, resnet2, upsample = module
    # First resnet takes 2x channels (skip connection)
    print(f"  Block {i}: {resnet1.conv1.block[0].in_channels} (w/ skip) → {resnet2.conv1.block[0].out_channels}")
    print(f"           Upsample: {type(upsample).__name__}")

print(f"\n--- Final Conv ---")
print(f"  {unet.final_conv}")

In [ ]:
# Deep dive: FiLM conditioning in a single residual block
# This is HOW the observations influence the denoising

res_block = unet.down_modules[0][0]  # First residual block
print("=== FiLM in a Residual Block ===")
print(f"use_film_scale_modulation: {res_block.use_film_scale_modulation}")
print(f"Output channels: {res_block.out_channels}")
print(f"\nFiLM encoder: {res_block.cond_encoder}")

# The FiLM encoder maps the conditioning to scale + bias
cond_out_dim = res_block.cond_encoder[1].out_features
print(f"\nFiLM output dim: {cond_out_dim}")
print(f"  = {res_block.out_channels} (scale) + {res_block.out_channels} (bias)")

print("\n--- What happens in forward() ---")
print("1. x = conv1(x)                     # First convolution")
print("2. cond_embed = FiLM_encoder(cond)   # [B, 2*channels, 1]")
print("3. scale = cond_embed[:, :channels]  # First half = scale")
print("4. bias  = cond_embed[:, channels:]  # Second half = bias")
print("5. x = scale * x + bias              # THE KEY OPERATION")
print("6. x = conv2(x)                      # Second convolution")
print("7. x = x + residual(input)           # Skip connection")

In [ ]:
# Let's actually run the U-Net on a noisy trajectory and see what comes out

with torch.no_grad():
    # Create a dummy noisy trajectory (what inference starts with)
    noisy_actions = torch.randn(1, 16, 2, device=device)  # Pure Gaussian noise
    timestep = torch.tensor([99], device=device)  # Start at max noise

    print("=== U-Net Forward Pass ===")
    print(f"Input (noisy actions): {noisy_actions.shape}, mean={noisy_actions.mean():.3f}, std={noisy_actions.std():.3f}")
    print(f"Timestep: {timestep.item()}")
    print(f"Global conditioning: {global_cond.shape}")

    # Run U-Net
    predicted_noise = unet(noisy_actions, timestep, global_cond=global_cond)
    print(f"\nOutput (predicted noise): {predicted_noise.shape}")
    print(f"  mean={predicted_noise.mean():.3f}, std={predicted_noise.std():.3f}")
    print(f"\nThe U-Net predicts what noise was added, so we can subtract it!")

### 3E: The DDPM Forward and Reverse Process

**Forward process** (training): Take a clean action trajectory, gradually add noise over 100 steps.

**Reverse process** (inference): Start from pure noise, iteratively denoise using the U-Net.

The noise schedule determines HOW MUCH noise is added at each step. LeRobot uses `squaredcos_cap_v2` — a cosine schedule that adds noise more gradually than linear schedules.

In [ ]:
# Visualize the noise schedule
scheduler = policy.diffusion.noise_scheduler

# alphas_cumprod tells us how much of the original signal remains at each timestep
alphas_cumprod = scheduler.alphas_cumprod.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Alpha cumprod (signal preservation)
axes[0].plot(alphas_cumprod, 'b-', linewidth=2)
axes[0].set_xlabel('Diffusion timestep t')
axes[0].set_ylabel('ᾱ_t (signal preserved)')
axes[0].set_title('How much original signal remains')
axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='50% signal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Noise level (1 - alpha_cumprod)
noise_level = 1 - alphas_cumprod
axes[1].plot(noise_level, 'r-', linewidth=2)
axes[1].set_xlabel('Diffusion timestep t')
axes[1].set_ylabel('1 - ᾱ_t (noise level)')
axes[1].set_title('How much noise has been added')
axes[1].grid(True, alpha=0.3)

# SNR (signal-to-noise ratio)
snr = alphas_cumprod / (1 - alphas_cumprod + 1e-8)
axes[2].plot(np.log10(snr + 1e-8), 'g-', linewidth=2)
axes[2].set_xlabel('Diffusion timestep t')
axes[2].set_ylabel('log₁₀(SNR)')
axes[2].set_title('Signal-to-Noise Ratio (log scale)')
axes[2].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='SNR = 1')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'DDPM Noise Schedule: {scheduler.config.beta_schedule}', fontsize=14)
plt.tight_layout()
plt.show()

print(f"At t=0:  {alphas_cumprod[0]:.4f} signal preserved (almost clean)")
print(f"At t=50: {alphas_cumprod[50]:.4f} signal preserved")
print(f"At t=99: {alphas_cumprod[99]:.4f} signal preserved (almost pure noise)")

In [ ]:
# Visualize the FORWARD process: progressively adding noise to a clean trajectory

# Get a clean expert trajectory
sample = dataset[1000]
clean_actions = sample['action'].unsqueeze(0).to(device)  # [1, 16, 2]
print(f"Clean trajectory shape: {clean_actions.shape}")

# Add noise at different timesteps
timesteps_to_show = [0, 10, 25, 50, 75, 99]
noise = torch.randn_like(clean_actions)  # Same noise sample for all

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(20, 4))

for i, t in enumerate(timesteps_to_show):
    with torch.no_grad():
        t_tensor = torch.tensor([t], device=device)
        noisy = scheduler.add_noise(clean_actions, noise, t_tensor)
        noisy_np = noisy[0].cpu().numpy()
        clean_np = clean_actions[0].cpu().numpy()

    axes[i].plot(clean_np[:, 0], clean_np[:, 1], 'g.-', alpha=0.3, label='Clean')
    axes[i].plot(noisy_np[:, 0], noisy_np[:, 1], 'r.-', linewidth=2, label='Noisy')
    axes[i].set_title(f't = {t}\nᾱ = {alphas_cumprod[t]:.3f}')
    axes[i].set_xlim(-2, 2)
    axes[i].set_ylim(-2, 2)
    axes[i].set_aspect('equal')
    axes[i].grid(True, alpha=0.3)
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('Forward Process: Adding noise to a clean action trajectory', fontsize=14)
plt.tight_layout()
plt.show()

print("At t=0: trajectory is almost unchanged")
print("At t=99: trajectory is pure noise — no trace of the original!")
print("\nThe U-Net must learn to REVERSE this process.")

### 3F: Training Loss — What the Model Actually Learns

The training procedure is beautifully simple:

1. Take a **clean action trajectory** from the dataset
2. Sample a **random timestep** `t ~ Uniform(0, 99)`
3. Sample **random noise** `ε ~ N(0, I)`
4. Create the **noisy trajectory**: `x_t = √(ᾱ_t) · x_0 + √(1-ᾱ_t) · ε`
5. Ask the U-Net to **predict the noise**: `ε̂ = UNet(x_t, t, conditioning)`
6. **Loss = MSE(ε̂, ε)** — how well did it predict the actual noise?

That's it. The whole training loop.

In [ ]:
# Let's do ONE training step manually to see everything

from lerobot.policies.factory import make_pre_post_processors
import torch.nn.functional as F

preprocessor, postprocessor = make_pre_post_processors(cfg, dataset_stats=metadata.stats)

# Get a batch from the dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)
batch = next(iter(dataloader))
batch = preprocessor(batch)

print("=== One Training Step (Manual) ===")
print(f"\nBatch keys: {list(batch.keys())}")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

# The policy's forward() does all of this internally,
# but let's see the loss
policy.train()
loss, _ = policy.forward(batch)
print(f"\nTraining loss: {loss.item():.4f}")
print("(This is MSE between predicted noise and actual noise)")

---
## Part 4: Training Diffusion Policy on PushT

Now let's train for real! With 5000 steps on a T4 GPU, this takes about 1-2 hours.

We'll train for fewer steps first to see progress quickly, then you can extend.

In [ ]:
# Fresh policy for training
policy = DiffusionPolicy(cfg)
policy.train()
policy.to(device)

preprocessor, postprocessor = make_pre_post_processors(cfg, dataset_stats=metadata.stats)

# Training setup
TRAINING_STEPS = 5000   # Increase to 10000+ for better results
BATCH_SIZE = 64
LR = 1e-4
LOG_FREQ = 100

optimizer = torch.optim.Adam(policy.parameters(), lr=LR)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Training for {TRAINING_STEPS} steps with batch_size={BATCH_SIZE}")
print(f"Dataset: {len(dataset)} samples, {len(dataloader)} batches per epoch")
print(f"Epochs needed: ~{TRAINING_STEPS / len(dataloader):.1f}")

In [ ]:
# Training loop
import time

losses = []
step = 0
done = False
start_time = time.time()

while not done:
    for batch in dataloader:
        batch = preprocessor(batch)

        # Forward pass
        loss, _ = policy.forward(batch)

        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        losses.append(loss.item())

        if step % LOG_FREQ == 0:
            elapsed = time.time() - start_time
            avg_loss = np.mean(losses[-LOG_FREQ:]) if len(losses) >= LOG_FREQ else np.mean(losses)
            steps_per_sec = (step + 1) / elapsed
            eta = (TRAINING_STEPS - step) / max(steps_per_sec, 0.01)
            print(f"Step {step:5d}/{TRAINING_STEPS} | Loss: {loss.item():.4f} | Avg: {avg_loss:.4f} | {steps_per_sec:.1f} steps/s | ETA: {eta/60:.1f}min")

        step += 1
        if step >= TRAINING_STEPS:
            done = True
            break

total_time = time.time() - start_time
print(f"\nTraining complete! {TRAINING_STEPS} steps in {total_time/60:.1f} minutes")
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Plot training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw loss
axes[0].plot(losses, alpha=0.3, color='blue')
# Smoothed loss
window = min(100, len(losses)//10)
if window > 1:
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2, label=f'Smoothed (window={window})')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log scale
axes[1].semilogy(losses, alpha=0.3, color='blue')
if window > 1:
    axes[1].semilogy(range(window-1, len(losses)), smoothed, 'r-', linewidth=2)
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('MSE Loss (log scale)')
axes[1].set_title('Training Loss (log scale)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
output_dir = Path("outputs/pusht_diffusion")
output_dir.mkdir(parents=True, exist_ok=True)

policy.save_pretrained(output_dir)
preprocessor.save_pretrained(output_dir)
postprocessor.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

---
## Part 5: Evaluation — Does It Actually Work?

Let's run our trained policy in the PushT environment and see if it can push the T-block onto the target!

**Inference process**:
1. Get observation from environment
2. Feed to policy → generates 16 actions via 100-step denoising
3. Execute first 8 actions (receding horizon)
4. When 8 actions are consumed, re-plan from current observation
5. Repeat until episode ends

In [ ]:
# Evaluate the trained policy

policy.eval()

def evaluate_policy(policy, preprocessor, postprocessor, n_episodes=5, max_steps=300, seed=42):
    """Run the policy in PushT and record videos."""
    env = gym.make(
        "gym_pusht/PushT-v0",
        obs_type="pixels_agent_pos",
        render_mode="rgb_array",
    )

    all_rewards = []
    all_frames = []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        policy.reset()  # Clear action queue!

        frames = [env.render()]
        total_reward = 0

        for step in range(max_steps):
            # Build observation dict for the policy
            state = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)
            image = torch.from_numpy(obs['pixels']).float().permute(2, 0, 1).unsqueeze(0).to(device) / 255.0

            obs_dict = {
                'observation.state': state,
                'observation.image': image,
            }
            obs_dict = preprocessor(obs_dict)

            # Get action from policy
            with torch.no_grad():
                action = policy.select_action(obs_dict)

            action = postprocessor(action)
            action_np = action.squeeze(0).cpu().numpy()

            # Step environment
            obs, reward, terminated, truncated, info = env.step(action_np)
            total_reward += reward

            if step % 3 == 0:
                frames.append(env.render())

            if terminated or truncated:
                break

        all_rewards.append(total_reward)
        all_frames.append(frames)
        print(f"  Episode {ep+1}: reward = {total_reward:.2f}, steps = {step+1}")

    env.close()
    return all_rewards, all_frames

print("Running evaluation...")
rewards, episode_frames = evaluate_policy(policy, preprocessor, postprocessor, n_episodes=5)
print(f"\nMean reward: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")

In [ ]:
# Show the best episode as a video
best_ep = np.argmax(rewards)
print(f"Showing best episode ({best_ep+1}) with reward {rewards[best_ep]:.2f}")
media.show_video(episode_frames[best_ep], fps=10, title=f"Trained Diffusion Policy (reward={rewards[best_ep]:.2f})")

In [ ]:
# Side-by-side: random policy vs trained policy
# Show first and last frames

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

best_frames = episode_frames[best_ep]
axes[0].imshow(best_frames[0])
axes[0].set_title("Start of episode")
axes[0].axis('off')

axes[1].imshow(best_frames[-1])
axes[1].set_title(f"End of episode (reward={rewards[best_ep]:.2f})")
axes[1].axis('off')

plt.suptitle('Trained Diffusion Policy on PushT', fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 6: Visualizing the Denoising Process

This is the most illuminating visualization — watch noise transform into a coherent action plan!

We'll manually run the denoising loop and capture intermediate trajectories.

In [ ]:
# Visualize the denoising process step by step
import einops

policy.eval()

# Get an observation from the environment
env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")
obs, info = env.reset(seed=42)
env.close()

# Build observation and compute global conditioning
state = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)  # [1, 2]
image = torch.from_numpy(obs['pixels']).float().permute(2, 0, 1).unsqueeze(0).to(device) / 255.0  # [1, 3, H, W]

obs_dict = {'observation.state': state, 'observation.image': image}
obs_dict = preprocessor(obs_dict)

# We need to manually prepare the conditioning (normally select_action handles this)
# Stack the observation to simulate 2 timesteps (duplicate for simplicity)
batch_for_cond = {
    'observation.state': obs_dict['observation.state'].unsqueeze(1).repeat(1, 2, 1),  # [1, 2, state_dim]
    'observation.images': obs_dict['observation.image'].unsqueeze(1).unsqueeze(2).repeat(1, 2, 1, 1, 1, 1),  # [1, 2, 1, C, H, W]
}

with torch.no_grad():
    global_cond = policy.diffusion._prepare_global_conditioning(batch_for_cond)
    print(f"Global conditioning shape: {global_cond.shape}")

    # Now manually run the denoising loop, capturing intermediate states
    noise_scheduler = policy.diffusion.noise_scheduler
    unet = policy.diffusion.unet

    # Start from pure noise
    action_dim = cfg.action_feature.shape[0]
    sample = torch.randn(1, cfg.horizon, action_dim, device=device)

    noise_scheduler.set_timesteps(policy.diffusion.num_inference_steps)

    # Capture trajectories at specific steps
    capture_at = {99, 90, 75, 50, 25, 10, 5, 2, 0}  # Timesteps to capture
    captured = {}

    for t in noise_scheduler.timesteps:
        t_val = t.item()
        if t_val in capture_at:
            captured[t_val] = sample[0].cpu().numpy().copy()

        model_output = unet(
            sample,
            torch.full((1,), t, dtype=torch.long, device=device),
            global_cond=global_cond,
        )
        sample = noise_scheduler.step(model_output, t, sample).prev_sample

    # Capture the final denoised result
    captured['final'] = sample[0].cpu().numpy().copy()

print(f"Captured {len(captured)} intermediate states")

In [ ]:
# Plot the denoising progression
keys_in_order = sorted([k for k in captured.keys() if k != 'final'], reverse=True) + ['final']

n_plots = len(keys_in_order)
fig, axes = plt.subplots(2, (n_plots + 1) // 2, figsize=(20, 8))
axes = axes.flatten()

for i, key in enumerate(keys_in_order):
    traj = captured[key]
    axes[i].plot(traj[:, 0], traj[:, 1], 'b.-', linewidth=1.5, markersize=5)
    axes[i].plot(traj[0, 0], traj[0, 1], 'go', markersize=10)   # Start
    axes[i].plot(traj[-1, 0], traj[-1, 1], 'rs', markersize=10) # End

    if key == 'final':
        axes[i].set_title(f'FINAL (denoised)', fontsize=11, fontweight='bold', color='green')
    else:
        axes[i].set_title(f't = {key}', fontsize=11)

    axes[i].set_aspect('equal')
    axes[i].grid(True, alpha=0.3)

# Hide any extra subplots
for j in range(len(keys_in_order), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Denoising Process: Pure Noise → Coherent Action Trajectory\n'
             '(green = start, red = end of trajectory)', fontsize=14)
plt.tight_layout()
plt.show()

print("Watch how the random noise gradually organizes into a smooth, coherent trajectory!")
print("Early steps (t=99): random scatter")
print("Middle steps (t=50): rough direction emerges")
print("Late steps (t=5→final): fine-grained trajectory refinement")

---
## Part 7: The Multimodality Advantage

This is THE key insight of Diffusion Policy. Because we start from random noise, **different noise samples produce different valid trajectories** from the same observation.

A standard MSE regression policy would average all valid solutions and produce garbage. Diffusion naturally handles multimodality.

In [ ]:
# Run inference 20 times from the SAME observation with different noise
# Each run should produce a different but valid trajectory

n_samples = 20
all_trajectories = []

with torch.no_grad():
    for i in range(n_samples):
        # Start from different random noise each time
        noise = torch.randn(1, cfg.horizon, action_dim, device=device)

        noise_scheduler.set_timesteps(policy.diffusion.num_inference_steps)
        sample = noise.clone()

        for t in noise_scheduler.timesteps:
            model_output = unet(
                sample,
                torch.full((1,), t, dtype=torch.long, device=device),
                global_cond=global_cond,
            )
            sample = noise_scheduler.step(model_output, t, sample).prev_sample

        all_trajectories.append(sample[0].cpu().numpy())

print(f"Generated {n_samples} trajectories from the same observation")

# Plot all trajectories overlaid
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# All trajectories overlaid
cmap = plt.cm.viridis(np.linspace(0, 1, n_samples))
for i, traj in enumerate(all_trajectories):
    axes[0].plot(traj[:, 0], traj[:, 1], '.-', color=cmap[i], alpha=0.6, linewidth=1.5)
axes[0].set_title(f'{n_samples} Different Action Trajectories\n(same observation, different noise seeds)')
axes[0].set_xlabel('Action x')
axes[0].set_ylabel('Action y')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Standard deviation at each timestep
all_traj_array = np.stack(all_trajectories)  # [n_samples, 16, 2]
std_per_step = all_traj_array.std(axis=0)  # [16, 2]
mean_std = np.mean(std_per_step, axis=1)  # [16]

axes[1].bar(range(16), mean_std, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Action step in horizon')
axes[1].set_ylabel('Std deviation across samples')
axes[1].set_title('Trajectory Diversity Per Step\n(higher = more multimodal)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: the model generates DIVERSE trajectories, not just one!")
print("This is impossible with a standard regression policy (which would average modes).")
print("\nNotice: later steps in the horizon tend to be more diverse")
print("(the model is less certain about the distant future).")

---
## Part 8: Ablation Experiments

Now let's test which components actually matter. Each ablation changes ONE thing and measures the effect.

**How to use this section**: Pick 2-3 ablations, train each for the same number of steps, compare final loss and evaluation reward.

### Ablation Menu
| # | Experiment | What Changes | Expected Effect |
|---|-----------|-------------|----------------|
| A | DDIM scheduler | 10 inference steps instead of 100 | 10x faster inference, similar quality? |
| B | Fewer keypoints | 8 keypoints instead of 32 | Spatial precision drops? |
| C | Smaller U-Net | down_dims=(256,512,1024) | Faster training, worse quality? |
| D | No FiLM scale | use_film_scale_modulation=False | Bias-only conditioning |
| E | Predict sample | prediction_type="sample" | Different loss target |

In [ ]:
# Helper function to train an ablation quickly

def train_ablation(config_overrides: dict, steps=2000, label="ablation"):
    """Train a diffusion policy variant and return the loss curve."""
    # Create config with overrides
    abl_cfg = DiffusionConfig(
        input_features=input_features,
        output_features=output_features,
        **config_overrides
    )

    abl_policy = DiffusionPolicy(abl_cfg)
    abl_policy.train()
    abl_policy.to(device)

    abl_pre, abl_post = make_pre_post_processors(abl_cfg, dataset_stats=metadata.stats)

    optimizer = torch.optim.Adam(abl_policy.parameters(), lr=LR)

    losses = []
    step = 0
    done = False
    while not done:
        for batch in dataloader:
            batch = abl_pre(batch)
            loss, _ = abl_policy.forward(batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            losses.append(loss.item())
            step += 1
            if step % 500 == 0:
                print(f"  [{label}] Step {step}/{steps} | Loss: {loss.item():.4f}")
            if step >= steps:
                done = True
                break

    n_params = sum(p.numel() for p in abl_policy.parameters())
    print(f"  [{label}] Done. Final loss: {losses[-1]:.4f}, Params: {n_params:,}")
    return losses, abl_policy, abl_pre, abl_post

print("Helper function ready. Run ablations below.")

In [ ]:
# ============================================================
# Ablation A: DDIM with 10 inference steps instead of DDPM 100
# This doesn't change training — only inference speed!
# ============================================================

print("=== Ablation A: DDIM (10 inference steps) vs DDPM (100 steps) ===")
print("Training is identical — we only change inference.\n")

# Use the ALREADY TRAINED policy from Part 4
# Just swap the scheduler at inference time

from lerobot.policies.diffusion.modeling_diffusion import _make_noise_scheduler

policy.eval()

# Time DDPM (100 steps)
import time

policy.diffusion.noise_scheduler = _make_noise_scheduler(
    "DDPM",
    num_train_timesteps=100,
    beta_start=0.0001, beta_end=0.02,
    beta_schedule="squaredcos_cap_v2",
    clip_sample=True, clip_sample_range=1.0,
    prediction_type="epsilon",
)
policy.diffusion.num_inference_steps = 100

start = time.time()
with torch.no_grad():
    for _ in range(5):
        noise = torch.randn(1, cfg.horizon, action_dim, device=device)
        _ = policy.diffusion.conditional_sample(1, global_cond=global_cond, noise=noise)
ddpm_time = (time.time() - start) / 5
print(f"DDPM (100 steps): {ddpm_time*1000:.1f} ms per trajectory")

# Time DDIM (10 steps)
policy.diffusion.noise_scheduler = _make_noise_scheduler(
    "DDIM",
    num_train_timesteps=100,
    beta_start=0.0001, beta_end=0.02,
    beta_schedule="squaredcos_cap_v2",
    clip_sample=True, clip_sample_range=1.0,
    prediction_type="epsilon",
)
policy.diffusion.num_inference_steps = 10

start = time.time()
with torch.no_grad():
    for _ in range(5):
        noise = torch.randn(1, cfg.horizon, action_dim, device=device)
        _ = policy.diffusion.conditional_sample(1, global_cond=global_cond, noise=noise)
ddim_time = (time.time() - start) / 5
print(f"DDIM (10 steps):  {ddim_time*1000:.1f} ms per trajectory")
print(f"\nSpeedup: {ddpm_time/ddim_time:.1f}x faster with DDIM!")

# Reset to DDPM for future cells
policy.diffusion.noise_scheduler = _make_noise_scheduler(
    "DDPM",
    num_train_timesteps=100,
    beta_start=0.0001, beta_end=0.02,
    beta_schedule="squaredcos_cap_v2",
    clip_sample=True, clip_sample_range=1.0,
    prediction_type="epsilon",
)
policy.diffusion.num_inference_steps = 100

In [ ]:
# ============================================================
# Ablation B: Fewer keypoints (8 vs 32)
# Does spatial precision matter?
# ============================================================

print("=== Ablation B: 8 Keypoints vs 32 Keypoints ===")
print("Training a new model with only 8 spatial keypoints...\n")

losses_8kp, _, _, _ = train_ablation(
    {"spatial_softmax_num_keypoints": 8},
    steps=2000,
    label="8-keypoints"
)

In [ ]:
# ============================================================
# Ablation C: Smaller U-Net
# ============================================================

print("=== Ablation C: Smaller U-Net ===")
print("down_dims=(256, 512, 1024) instead of (512, 1024, 2048)...\n")

losses_small, _, _, _ = train_ablation(
    {"down_dims": (256, 512, 1024)},
    steps=2000,
    label="small-unet"
)

In [ ]:
# Compare ablation training curves
fig, ax = plt.subplots(figsize=(10, 6))

window = 50

# Baseline (from Part 4)
baseline_2k = losses[:2000]  # First 2000 steps of full training
if len(baseline_2k) > window:
    smoothed_base = np.convolve(baseline_2k, np.ones(window)/window, mode='valid')
    ax.plot(smoothed_base, label='Baseline (32 kp, large UNet)', linewidth=2)

if len(losses_8kp) > window:
    smoothed_8kp = np.convolve(losses_8kp, np.ones(window)/window, mode='valid')
    ax.plot(smoothed_8kp, label='8 keypoints', linewidth=2)

if len(losses_small) > window:
    smoothed_small = np.convolve(losses_small, np.ones(window)/window, mode='valid')
    ax.plot(smoothed_small, label='Small U-Net', linewidth=2)

ax.set_xlabel('Training step')
ax.set_ylabel('MSE Loss')
ax.set_title('Ablation Comparison (2000 steps each)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

---
## Part 9: Using a Pretrained Model (Instant Gratification)

If your training hasn't converged yet (or you want to see what a fully trained model looks like), you can load the official pretrained checkpoint from HuggingFace.

In [ ]:
# Load the official pretrained model
pretrained_policy = DiffusionPolicy.from_pretrained("lerobot/diffusion_pusht")
pretrained_policy.eval()
pretrained_policy.to(device)

# Load matching preprocessor/postprocessor
pre_pt, post_pt = make_pre_post_processors(
    pretrained_policy.config,
    "lerobot/diffusion_pusht",
    dataset_stats=metadata.stats,
)

print("Pretrained model loaded! Evaluating...")
rewards_pt, frames_pt = evaluate_policy(pretrained_policy, pre_pt, post_pt, n_episodes=5)
print(f"\nPretrained model mean reward: {np.mean(rewards_pt):.2f} ± {np.std(rewards_pt):.2f}")

In [ ]:
# Show the pretrained model in action
best_ep_pt = np.argmax(rewards_pt)
media.show_video(
    frames_pt[best_ep_pt], fps=10,
    title=f"Pretrained Diffusion Policy (reward={rewards_pt[best_ep_pt]:.2f})"
)

---
## Exercises

### Exercise 1: Observation Ablation
Train a policy with `n_obs_steps=1` (only current frame, no history). Does temporal context matter?

### Exercise 2: Horizon Ablation
Change `horizon=8, n_action_steps=4`. Does a shorter planning horizon hurt?

### Exercise 3: Visualize Keypoint Motion
Record the 32 SpatialSoftmax keypoints at every step during an evaluation episode. Create a video showing keypoints overlaid on the image. Do they track task-relevant objects?

### Exercise 4: Compare with Mean Trajectory
From Part 7 (multimodality), compute the **mean** of the 20 sampled trajectories. Overlay it on the individual trajectories. This is what a naive regression policy would predict. How different is it?

### Exercise 5 (Challenge): State-Only Policy
Modify the config to use `obs_type="environment_state_agent_pos"` (no images, just state vectors). How does performance compare? When would you want vision vs state?

---

## Summary

| Component | Purpose | Key Numbers |
|-----------|---------|-------------|
| ResNet18 + SpatialSoftmax | Extract spatial keypoints from images | 32 keypoints × 2 coords = 64d/image |
| Sinusoidal + MLP | Encode diffusion timestep | 128d |
| Global conditioning | Everything the U-Net needs to know | ~260d (varies with state dim) |
| 1D U-Net | Predict noise from noisy actions + conditioning | ~26M params |
| FiLM | Inject conditioning at every U-Net layer | scale × hidden + bias |
| DDPM | 100-step denoising from noise → actions | |
| Action chunking | Predict 16, execute 8, re-plan | Temporal smoothness |

**Next week**: We'll scale to bimanual manipulation with gym-aloha (ALOHA simulation)!